# 03 — Pipeline de Production : Kafka + Spark Streaming + ML

**Objectif** : Assembler tout ce qui a ete developpe en Phase 1 dans un pipeline de production temps reel.

**Architecture** :

Kafka Producer → Topic `tweets-raw-{username}` → Spark Structured Streaming → Modele ML → HDFS → Dashboard

**Prerequis** :
- Kafka demarre (verifiez dans Kafka-UI : http://10.15.61.20:8080)
- Modele ML sauvegarde sur HDFS (`hdfs:///models/sentiment_model`) — cf. notebook 02
- Notebook `00_Setup.ipynb` execute (kafka-python installe)

In [1]:
import sys, os

# Ajouter ~/python_packages/ au sys.path (packages installes par 00_Setup.ipynb)
_user_pkg_dir = os.path.expanduser("~/python_packages")
if _user_pkg_dir not in sys.path:
    sys.path.insert(0, _user_pkg_dir)

username = os.environ.get('USER', 'unknown')
topic = f"tweets-raw-{username}"
print(f"Utilisateur : {username}")
print(f"Topic Kafka : {topic}")

Utilisateur : ldixne01
Topic Kafka : tweets-raw-ldixne01


## 1. Verification de Kafka

In [2]:
from kafka.admin import KafkaAdminClient

try:
    admin = KafkaAdminClient(bootstrap_servers='kafka:9092', request_timeout_ms=5000)
    topics = admin.list_topics()
    print(f"[OK] Connexion Kafka reussie")
    print(f"Topics existants ({len(topics)}) : {', '.join(topics[:10]) if topics else 'aucun'}")
    admin.close()
except Exception as e:
    print(f"[ERREUR] Impossible de se connecter a Kafka : {e}")
    print("Verifiez que le stack Kafka est demarre.")

[OK] Connexion Kafka reussie
Topics existants (28) : tweets-raw-lmauri05, tweets-raw-dzhou01, tweets-raw-ttessi05, tweet_raw_vdujour, votre_topic_perso, tweets-raw-mfaget, tweets-raw-vdujour, mon_topic_tweets, tweets-raw-ymouchar, tweets-raw-qchabot


## 2. Producteur Kafka minimal

In [3]:
import json
from datetime import datetime
from kafka import KafkaProducer

# ============================================================
# TODO [Exercice 2.1] : Producteur Kafka minimal
#
# Consigne : Creez un KafkaProducer connecte a 'kafka:9092'
#            et envoyez 10 tweets de test vers votre topic personnel.
#            Chaque tweet doit etre un dict avec les cles :
#            sentiment, tweet_id, date, user, text, produced_at
#
# Indice   : KafkaProducer(bootstrap_servers='kafka:9092',
#              value_serializer=lambda v: json.dumps(v).encode('utf-8'),
#              key_serializer=lambda k: k.encode('utf-8') if k else None)
#            producer.send(topic, key=tweet["user"], value=tweet)
#            N'oubliez pas producer.flush() et producer.close()
#
# Attendu  : 10 messages visibles dans Kafka-UI (http://10.15.61.20:8080)
# ============================================================

import json
import time
from datetime import datetime
from kafka import KafkaProducer

# Configuration du Producteur
producer = KafkaProducer(
    bootstrap_servers='kafka:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'), # Sérialisation JSON -> Bytes
    key_serializer=lambda k: k.encode('utf-8') if k else None
)

# Création de 10 tweets factices pour tester
test_tweets = [
    {"sentiment": 4, "tweet_id": str(i), "date": str(datetime.now()), 
     "user": username, "text": f"Ceci est le tweet de test numero {i} ! J'adore Spark.", 
     "produced_at": str(datetime.now())}
    for i in range(10)
]

print(f"Envoi de 10 messages vers {topic}...")

for tweet in test_tweets:
    # On utilise le username comme clé (partitionnement par utilisateur)
    producer.send(topic, key=tweet["user"], value=tweet)
    
# Important : s'assurer que tout est envoyé avant de fermer
producer.flush()
producer.close()

print("Envoi terminé.")

Envoi de 10 messages vers tweets-raw-ldixne01...
Envoi terminé.


## 3. Consommateur Kafka minimal

In [4]:
from kafka import KafkaConsumer

# ============================================================
# TODO [Exercice 3.1] : Consommateur Kafka minimal
#
# Consigne : Creez un KafkaConsumer qui lit votre topic depuis le debut
#            et affichez les messages recus.
#
# Indice   : KafkaConsumer(topic, bootstrap_servers='kafka:9092',
#              auto_offset_reset='earliest',
#              value_deserializer=lambda m: json.loads(m.decode('utf-8')),
#              consumer_timeout_ms=5000)
#            Iterez sur consumer pour lire les messages
#
# Attendu  : Les 10 tweets envoyes a l'exercice precedent s'affichent
# ============================================================

from kafka import KafkaConsumer
import json

print(f"Lecture du topic {topic}...")

# Configuration du Consommateur
consumer = KafkaConsumer(
    topic,
    bootstrap_servers='kafka:9092',
    # Pour lire les messages qu'on vient d'envoyer
    auto_offset_reset='earliest', 
    # Désérialisation Bytes -> JSON
    value_deserializer=lambda m: json.loads(m.decode('utf-8')), 
    # Arrêter d'écouter après 5 secondes de silence (pour ne pas bloquer la cellule)
    consumer_timeout_ms=5000 
)

count = 0
for message in consumer:
    print(f"Reçu (offset {message.offset}): {message.value}")
    count += 1
    if count >= 10: 
        break

print(f"Lecture terminée. {count} messages lus.")
consumer.close()

Lecture du topic tweets-raw-ldixne01...
Reçu (offset 1198978): {'sentiment': 4, 'tweet_id': '0', 'date': '2026-02-08 22:38:25.837534', 'user': 'ldixne01', 'text': "Ceci est le tweet de test numero 0 ! J'adore Spark.", 'produced_at': '2026-02-08 22:38:25.837549'}
Reçu (offset 1198979): {'sentiment': 4, 'tweet_id': '1', 'date': '2026-02-08 22:38:25.837557', 'user': 'ldixne01', 'text': "Ceci est le tweet de test numero 1 ! J'adore Spark.", 'produced_at': '2026-02-08 22:38:25.837562'}
Reçu (offset 1198980): {'sentiment': 4, 'tweet_id': '2', 'date': '2026-02-08 22:38:25.837566', 'user': 'ldixne01', 'text': "Ceci est le tweet de test numero 2 ! J'adore Spark.", 'produced_at': '2026-02-08 22:38:25.837571'}
Reçu (offset 1198981): {'sentiment': 4, 'tweet_id': '3', 'date': '2026-02-08 22:38:25.837575', 'user': 'ldixne01', 'text': "Ceci est le tweet de test numero 3 ! J'adore Spark.", 'produced_at': '2026-02-08 22:38:25.837578'}
Reçu (offset 1198982): {'sentiment': 4, 'tweet_id': '4', 'date': '20

## 4. Spark Structured Streaming depuis Kafka

In [5]:
import os

# Définir les variables d'environnement AVANT de créer SparkSession
os.environ['http_proxy'] = 'http://proxy.iutn.univ-poitiers.fr:3128'
os.environ['https_proxy'] = 'http://proxy.iutn.univ-poitiers.fr:3128'
os.environ['HTTP_PROXY'] = 'http://proxy.iutn.univ-poitiers.fr:3128'
os.environ['HTTPS_PROXY'] = 'http://proxy.iutn.univ-poitiers.fr:3128'

# Configuration Java pour Ivy (téléchargement Maven)
os.environ['JAVA_TOOL_OPTIONS'] = (
    '-Dhttp.proxyHost=proxy.iutn.univ-poitiers.fr -Dhttp.proxyPort=3128 '
    '-Dhttps.proxyHost=proxy.iutn.univ-poitiers.fr -Dhttps.proxyPort=3128'
) 

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName(f"Production-{username}") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.instances", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession creee avec le connecteur Kafka")

Picked up JAVA_TOOL_OPTIONS: -Dhttp.proxyHost=proxy.iutn.univ-poitiers.fr -Dhttp.proxyPort=3128 -Dhttps.proxyHost=proxy.iutn.univ-poitiers.fr -Dhttps.proxyPort=3128
Picked up JAVA_TOOL_OPTIONS: -Dhttp.proxyHost=proxy.iutn.univ-poitiers.fr -Dhttp.proxyPort=3128 -Dhttps.proxyHost=proxy.iutn.univ-poitiers.fr -Dhttps.proxyPort=3128


:: loading settings :: url = jar:file:/opt/spark-3.5.3-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ldixne01/.ivy2/cache
The jars for the packages stored in: /home/ldixne01/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1673e768-34c3-4618-8c9b-aea396083d7f;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 414ms :: artifacts dl 12ms

SparkSession creee avec le connecteur Kafka


In [7]:
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# ============================================================
# TODO [Exercice 4.1] : Lecture du flux Kafka avec Spark
#
# Consigne : Definissez le schema JSON des messages Kafka, puis
#            utilisez spark.readStream pour lire depuis votre topic.
#            Parsez le JSON contenu dans la colonne 'value'.
#
# Indice   : Schema : StructType avec les champs
#            sentiment (IntegerType), tweet_id, date, user, text, produced_at (StringType)
#
#            spark.readStream.format("kafka")
#              .option("kafka.bootstrap.servers", "kafka:9092")
#              .option("subscribe", topic)
#              .option("startingOffsets", "latest")
#              .load()
#
#            Pour parser : from_json(col("value").cast("string"), schema)
#            Ajoutez event_time = to_timestamp(col("produced_at"))
#            Le resultat doit s'appeler 'df_parsed'
#
# Attendu  : Un DataFrame streaming avec les colonnes du tweet + event_time
# ============================================================

from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Définition du schéma JSON attendu de structure des tweets
schema = StructType([
    StructField("sentiment", IntegerType()),
    StructField("tweet_id", StringType()),
    StructField("date", StringType()),
    StructField("user", StringType()),
    StructField("text", StringType()),
    StructField("produced_at", StringType())
])

# Lecture du flux Kafka
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", topic) \
    .option("startingOffsets", "latest") \
    .load()

# Parsing des données = Transformation
# On convertit la colonne binaire 'value' en String, puis on parse le json
df_parsed = df_raw \
    .select(from_json(col("value").cast("string"), schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", to_timestamp(col("produced_at")))

# schéma 
print("Schéma du DataFrame parsé :")
df_parsed.printSchema()

Schéma du DataFrame parsé :
root
 |-- sentiment: integer (nullable = true)
 |-- tweet_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- user: string (nullable = true)
 |-- text: string (nullable = true)
 |-- produced_at: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



on a transformé le flux binaire brut de Kafka en un dataframe. Grâce au schéma, Spark interprète chaque message json pour extraire les colonnes

on a généré un event_time indispensable pour gérer l'ordre chronologique des tweets

## 5. Integration du modele ML

> **Attention au training-serving skew** : le nettoyage de texte doit etre **identique** a celui utilise lors de l'entrainement (notebook 02).

In [8]:
import re
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType as ST
from pyspark.ml import PipelineModel

# UDF de nettoyage IDENTIQUE au training (notebook 02)
# FOURNI — ne pas modifier !
def clean_text(text):
    if text is None: return ""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_text_udf = udf(clean_text, ST())

# ============================================================
# TODO [Exercice 5.1] : Integration du modele ML
#
# Consigne : 1. Appliquez le nettoyage sur df_parsed -> df_cleaned
#            2. Chargez le modele ML depuis HDFS
#            3. Appliquez le modele sur df_cleaned
#            4. Selectionnez les colonnes utiles dans df_predicted
#
# Indice   : df_cleaned = df_parsed.withColumn("clean_text", clean_text_udf(col("text")))
#            model = PipelineModel.load("hdfs:///models/sentiment_model")
#            df_predicted = model.transform(df_cleaned).select(...)
#
# Attendu  : Un DataFrame 'df_predicted' avec tweet_id, user, text,
#            event_time, sentiment, predicted_sentiment, clean_text
# ============================================================


# Application du nettoyage sur le flux
# On crée la colonne 'clean_text' que le modèle attend en entrée
df_cleaned = df_parsed.withColumn("clean_text", clean_text_udf(col("text")))

# Chargement du modèle ML depuis HDFS
# Note : Si ton modèle s'appelle différemment, change le chemin ici !
print("Chargement du modèle depuis : hdfs:///models/LDF_sentiment_model_stack_norm")
model = PipelineModel.load("hdfs:///models/LDF_sentiment_model_stack_norm")

# Application du modèle pour les prédictions
df_predicted_raw = model.transform(df_cleaned)

# Sélection et renommage des colonnes finales
df_predicted = df_predicted_raw.select(
    col("tweet_id"),
    col("user"),
    col("text"),
    col("event_time"),
    col("sentiment").alias("original_sentiment"), # Le vrai sentiment
    col("prediction").alias("predicted_sentiment"), # La prédiction du modèle
    col("clean_text")
)

print("Schéma du DataFrame avec prédictions :")
df_predicted.printSchema()

Chargement du modèle depuis : hdfs:///models/LDF_sentiment_model_stack_norm


Schéma du DataFrame avec prédictions :
root
 |-- tweet_id: string (nullable = true)
 |-- user: string (nullable = true)
 |-- text: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- original_sentiment: integer (nullable = true)
 |-- predicted_sentiment: double (nullable = false)
 |-- clean_text: string (nullable = true)



Nous avons chargé notre modèle de Machine Learning pré-entraîné depuis HDFS. Pour chaque nouveau tweet arrivant dans le flux, nous appliquons d'abord le même nettoyage que lors de l'entraînement (clean_text), puis le modèle effectue une prédiction (0 ou 1) qui est stockée dans la colonne predicted_sentiment.

## 6. Ecriture des resultats vers HDFS

In [9]:
# Chemin HDFS personnel pour les resultats
output_path = f"/user/ldixne01/streaming_results"
checkpoint_path = f"/user/ldixne01/streaming_checkpoints_v2"

# ============================================================
# TODO [Exercice 6.1] : Ecriture des predictions sur HDFS
#
# Consigne : Ecrivez les predictions en streaming vers HDFS au format JSON.
#            Utilisez un trigger de 10 secondes.
#            La query doit s'appeler 'query_predictions'.
#
# Indice   : df_predicted.select("tweet_id", "user", "text", "event_time", "predicted_sentiment")
#            .writeStream.format("json")
#            .option("path", output_path)
#            .option("checkpointLocation", checkpoint_path)
#            .trigger(processingTime="10 seconds")
#            .start()
#
# Attendu  : Des fichiers JSON apparaissent dans hdfs:///user/{username}/streaming_results/
# ============================================================

# On sélectionne les colonnes finales et on lance l'écriture
query_predictions = df_predicted \
    .select("tweet_id", "user", "text", "event_time", "predicted_sentiment") \
    .writeStream \
    .format("json") \
    .outputMode("append") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="10 seconds") \
    .queryName("query_predictions") \
    .start()

print(f"Ecriture des predictions vers : {output_path}")
print(f"\nMAINTENANT, lancez le producteur Kafka dans un terminal JupyterHub :")
print(f"  python ~/notebooks/scripts/kafka_producer.py --rate 50")

Ecriture des predictions vers : /user/ldixne01/streaming_results

MAINTENANT, lancez le producteur Kafka dans un terminal JupyterHub :
  python ~/notebooks/scripts/kafka_producer.py --rate 50


26/02/08 22:39:21 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 7. Verification des resultats

In [10]:
import time
time.sleep(30)  # Attendre que des donnees arrivent

# Lire les resultats ecrits sur HDFS
df_results = spark.read.json(output_path)
print(f"Nombre de tweets predits : {df_results.count()}")
print()

# Apercu
df_results.show(10, truncate=50)

# Distribution des predictions
print("=== Distribution des predictions ===")
df_results.groupBy("predicted_sentiment").count().show()

26/02/08 22:39:22 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/02/08 22:39:23 WARN DAGScheduler: Broadcasting large task binary with size 1207.4 KiB
26/02/08 22:39:28 WARN TaskSetManager: Lost task 0.0 in stage 34.0 (TID 34) (worker05 executor 1): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to /user/ldixne01/streaming_results.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:775)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:420)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$executeWrite$2(FileFormatWriter.scala:252)
	at org.apache.spark.scheduler.ResultTask.runTask(ResultTask.scala:93)
	at org.apache.spark.TaskContext.runTaskWithListeners(TaskContext.scala:166)
	at or

Nombre de tweets predits : 599484

+------------------------+-------------------+--------------------------------------------------+----------+--------------+
|              event_time|predicted_sentiment|                                              text|  tweet_id|          user|
+------------------------+-------------------+--------------------------------------------------+----------+--------------+
|2026-02-04T17:47:59.994Z|                1.0|Excited about Airsoft on saturday  Just bought ...|2016121599|      TuckerUK|
|2026-02-04T17:48:00.014Z|                0.0|@FlickL; aw no babe, I don't work for Allianz, ...|2326067373|        1conic|
|2026-02-04T17:48:00.034Z|                0.0|Had to leave and disconnect from the Iran rucku...|2260360564|  JohannaWhyte|
|2026-02-04T17:48:00.054Z|                1.0|says Follow me on Twitter... and I also Follow ...|2191432923|    angelkim17|
|2026-02-04T17:48:00.074Z|                1.0|Im a very open girl and if you remain respectfu...|

[Stage 41:===================================================>    (35 + 3) / 38]

+-------------------+------+
|predicted_sentiment| count|
+-------------------+------+
|                0.0|290646|
|                1.0|308838|
+-------------------+------+



La sortie annonce qu'on a traité 599 484 tweets, ce qu iest très proche des 600k tweets attendus pour la production. 

La distribution est équilibrée avec 290k négatifs et 308k positifs, donc c'eest bien le modèle ne prédit pas toujours la meme chose.

## 8. Monitoring du pipeline

In [11]:
# Statut de la query
print(f"Query active : {query_predictions.isActive}")
print(f"Dernier progres :")
progress = query_predictions.lastProgress
if progress:
    print(f"  Timestamp      : {progress.get('timestamp', 'N/A')}")
    print(f"  Lignes traitees: {progress.get('numInputRows', 'N/A')}")
    print(f"  Duree batch    : {progress.get('batchDuration', 'N/A')} ms")

Query active : False
Dernier progres :


Le isactive sert à vérifier que le processus est toujours en cours d'exécution.

Le numInputRows est le nombre de nouveaux tweets arrivés depuis le dernier déclenchement (10s). Comme c'est à 0, cela signifie que Spark a rattrapé tout le retard et attend de nouvelles données.

## 9. Arret

In [12]:
# Arreter la query de streaming
query_predictions.stop()
print("Query de streaming arretee.")

spark.stop()
print("SparkSession arretee.")

Query de streaming arretee.
SparkSession arretee.


## 10. Questions de reflexion

1. **TCP vs Kafka** : Comparez l'architecture TCP socket (Phase 1) avec Kafka (Phase 2). Quels sont les avantages et inconvenients de chacune ?

L'architecture TCP socket utilisée en phase 1 est une connexion synchrone ce qui signifie qu'elle est simple à mettre en place pour le développement, mais elle est très fragile car si Spark plante ou est absent, les données envoyées par le serveur sont immédiatement perdues.

À l'inverse, l'architecture Kafka de la phase 2 introduit un tampon asynchrone et persistant entre la source et le traitement. Les messages sont stockés sur le disque dans le topic kafka, ce qui permet à Spark de redémarrer après une panne et de reprendre la lecture là où il s'était arrêté, et cela garantie ainsi qu'aucune donnée n'est perdue.


2. **Back-pressure** : Que se passe-t-il si le consumer Spark est plus lent que le producer Kafka ? Comment Kafka gere-t-il cette situation par rapport au TCP socket ?

Si le consumer Spark est plus lent que le producer Kafka, Kafka absorbe la différence en stockant les messages excédentaires sur son disque. Spark accumule donc simplement du retard (le lag), qu'il pourra rattraper plus tard sans perdre de données (tant que la capacité de stockage de Kafka n'est pas atteinte !).

En comparaison, avec un socket TCP, le tampon réseau est limité en taille. Si Spark ne lit pas assez vite pour vider ce tampon, les nouveaux paquets entrants sont rejetés et les données sont perdues, car le serveur TCP n'a pas de mécanisme de stockage pour conserver l'historique.


3. **Passage a l'echelle** : Comment modifieriez-vous cette architecture pour traiter 1 million de tweets par seconde ? Identifiez les goulots d'etranglement.

Pour traiter un million de tweets par seconde, il faudrait modifier l'architecture en augmentant le parallélisme des deux côtés. Du côté de Kafka, il serait nécessaire de diviser le topic en plusieurs centaines de partitions pour répartir la charge d'écriture, et côté Spark, il faudrait ajouter de nombreux nœuds au cluster pour avoir autant d'exécuteurs que de partitions à lire en même temps.

Les principaux goulots d'étranglement seraient probablement l'inférence du modèle ML, qui consomme beaucoup de CPU pour chaque tweet, et l'écriture sur HDFS, car le système de fichiers distribué supporte mal la création de millions de petits fichiers JSON par seconde. Une solution serait alors peut-être d'utiliser des bases de données optimisées pour l'écriture rapide ou de simplifier le modèle de nettoyage de texte pour réduire le temps de traitement unitaire.